# Module 4: Online Evaluations

> Self-directed module, ~25 min.

**Online evaluations** run automatically against every new trace as it lands in your tracing project — same evaluator pattern as the offline evaluations in Module 3, just triggered on incoming runs instead of a fixed dataset.

LangSmith calls these **run rules**. The Python SDK doesn't expose them directly, so we wrap the REST endpoint with a helper at `utils/langsmith_rules.py`. Pass in: a project name, an LLM-as-judge prompt, an output schema. Get back: the rule ID and a deep link to inspect it in the UI.

Workflow: define the judge → register it as a run rule → generate new traces → query them back with their feedback attached.


## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import os
import time
from datetime import datetime, timedelta, timezone
from langsmith import Client, uuid7

from utils.config import load_active_agent, PROJECT_NAME

# Single source of truth for project name; ensure tracing is on.
os.environ["LANGSMITH_PROJECT"] = PROJECT_NAME
os.environ.setdefault("LANGSMITH_TRACING", "true")

client = Client()
agent = load_active_agent()

print(f"Project:  {PROJECT_NAME}")
print(f"Tracing:  {os.environ.get('LANGSMITH_TRACING')}")
print(f"Endpoint: {os.environ.get('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com')}")

# Print the project URL so the "Verify in LangSmith" cells below have a direct deep link.
try:
    _project = client.read_project(project_name=PROJECT_NAME)
    print(f"Project URL: {_project.url}")
except Exception:
    print(f"Project '{PROJECT_NAME}' will appear in LangSmith once the first invoke runs.")

from utils.langsmith_rules import create_run_rule, delete_run_rule


## Step 1 — Define the judge

Run rules support several evaluator types; **LLM-as-judge** is the most common. A judge needs two pieces: a **prompt** that instructs the judge model how to score, and a **JSON schema** that constrains the judge's output. LangSmith uses the schema to validate the response and attach the scores as feedback on the run.

Defining both upfront has two effects:

- The rule fires identically on every run — no model-side variance from response shape changes.
- Once registered, the prompt and schema become editable in the LangSmith UI. The SDK is the bootstrap; the UI is the source of truth from that point forward.

For the client research use case, score on three signals: it answered the question, it cited a source for any market or news claim, and it didn't fabricate a client identity (only `acme-pension`, `smith-family-office`, `techcorp-treasury` are real).


In [ ]:
judge_prompt = (
    "You score a client research assistant's response.\n"
    "\n"
    "Mark correctness=true only if ALL of the following hold:\n"
    "  1. The response answers the user's actual question.\n"
    "  2. Any market or company news claims include at least one source URL.\n"
    "  3. The response does not invent client identities. Only 'acme-pension', 'smith-family-office', "
    "and 'techcorp-treasury' are real clients; an unknown client should be reported as not found, not fabricated.\n"
    "\n"
    "Otherwise mark correctness=false. Give one short sentence of comment either way."
)

judge_schema = {
    "title": "correctness",
    "description": "Score whether the client research response is grounded, sourced, and honest about unknown clients.",
    "type": "object",
    "properties": {
        "correctness": {"type": "boolean", "description": "True if all three criteria are met"},
        "comment": {"type": "string", "description": "One short sentence explaining the score"},
    },
    "required": ["correctness", "comment"],
    "strict": True,
}


## Step 2 — Register the rule

`create_run_rule` posts the judge to LangSmith and binds it to the project. `sampling_rate=1.0` scores every run; lower it for high-volume projects where 100% scoring is cost-prohibitive. The filter `eq(is_root, true)` prevents the rule from re-scoring every child span (LLM, tool, subagent) — only the top-level run gets a `correctness` score.

<img src="../images/03-rule-editor.png" alt="LangSmith rule editor showing the judge prompt and schema" style="width: auto; max-height: 360px; border-radius: 8px;">


In [ ]:
online_rule = create_run_rule(
    client,
    project_name=PROJECT_NAME,
    display_name="client-research-online-correctness",
    sampling_rate=1.0,
    filter="eq(is_root, true)",
    llm_judge_prompt=judge_prompt,
    llm_judge_schema=judge_schema,
)

print("Rule ID:    ", online_rule["id"])
print("Open in UI: ", online_rule["url"])


Open the rule URL above. The prompt and schema can be edited there from now on — the SDK call was a one-time bootstrap, not a runtime contract. Iterating on the judge happens in the UI.


## Step 3 — Generate scored traces

Trigger four runs that exercise the rule's three criteria differently. Feedback is asynchronous: the rule fires within ~30 seconds of a run completing, so the runs land in the project first and accumulate scores after.


In [ ]:
trigger_prompts = [
    "Summarize Microsoft's most recent earnings. One paragraph.",
    "What's the risk profile of acme-pension?",
    "Look up the client 'unknown-fund' and tell me about their holdings.",
    "Research recent news on the energy sector. Search at most once.",
]

total = 0.0
for q in trigger_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].content[:120]}")

print(f"\nTotal: {total:.1f}s")
print(f"\nFeedback should appear in {online_rule['url']} within ~30 seconds.")


## Step 4 — Query scored runs

Once the judge has run, the feedback is attached as a labeled key/value on each scored trace. Pull them back with a feedback filter — both the passes and the failures are individually addressable.

If the count comes back lower than expected, the rule may still be processing; re-run the cell after another 15–30 seconds.

To inspect scored runs visually, open the **Project URL** printed in the Setup cell and review the feedback column within the `traces` tab.


In [ ]:
since = datetime.now(timezone.utc) - timedelta(hours=1)

scored_runs = list(client.list_runs(
    project_name=PROJECT_NAME,
    start_time=since,
    is_root=True,
    filter='eq(feedback_key, "correctness")',
    limit=20,
))
print(f"{len(scored_runs)} scored root run(s) in the last hour")
for r in scored_runs[:5]:
    print(f"  {r.id}  {r.name:25s}")

# Failed only (correctness < 0.5)
failed_runs = list(client.list_runs(
    project_name=PROJECT_NAME,
    start_time=since,
    is_root=True,
    filter='and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))',
    limit=20,
))
print(f"\n{len(failed_runs)} failed run(s) (correctness < 0.5)")


### Verify in LangSmith

Open the **Project URL** printed in the Setup cell and switch to the **Traces** tab — the four newest runs should now show a `correctness` column with a green check or red X. Clicking a run opens the trace; the judge's comment is in the **Feedback** panel.

<img src="../images/online_evals.png" alt="Scored trace with correctness feedback visible" style="width: auto; max-height: 360px; border-radius: 8px;">


## Cleanup

Leave the rule active — Module 5 (annotation queues) routes runs based on its feedback. Tear it down later with:

```python
delete_run_rule(client, online_rule["id"])
```


## Recap

| Topic | API |
|---|---|
| Define LLM-as-judge | prompt string + JSON Schema for the judge output |
| Register the rule | `create_run_rule(client, project_name=..., llm_judge_prompt=..., llm_judge_schema=...)` |
| Query scored runs | `client.list_runs(filter='eq(feedback_key, "correctness")')` |
| Failed runs only | `filter='and(eq(feedback_key, ...), lt(feedback_score, 0.5))'` |

Next: `05_annotation_queues.ipynb` — route the failed runs to a human reviewer.
